In [1]:
# =========================
# 0. Install dependencies
# =========================

!pip install -q transformers sentencepiece protobuf tiktoken scikit-learn pandas tqdm torch


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
#!git clone https://github.com/ayadssk/AIR_Group_Task.git
#%cd /content/AIR_Group_Task

In [4]:
# =========================
# 1. Imports
# =========================

import os
import re
import json
import sys
import shutil
import subprocess
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_cosine_schedule_with_warmup,
)

In [5]:
# =========================
# 2. Paths and config
# =========================

BASE = "/content/drive/MyDrive/ColabNotebooks/AIR_Group_Task"
os.chdir(BASE)

BASE_MODEL   = "roberta-base"
MAX_LENGTH   = 128
BATCH_SIZE   = 4
EPOCHS       = 3
LR           = 1e-5
RANDOM_STATE = 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Working dir:", os.getcwd())
print("Device:", device)

# ── Language registry ─────────────────────────────────────────────────────
LANGUAGES = [
    {
        "lang"       : "english",
        "train_path" : f"{BASE}/data/english/english_train.json",
        "val_path"   : f"{BASE}/data/english/clef2026_gpt4_o_mini_val.json",
        "train_jsonl": f"{BASE}/output/training_data_for_RM/english_train_with_evidence.jsonl",
    },
    {
        "lang"       : "spanish",
        "train_path" : f"{BASE}/data/spanish/spanish_train.json",
        "val_path"   : f"{BASE}/data/spanish/spanish_val.json",
        "train_jsonl": f"{BASE}/output/training_data_for_RM/spanish_train_with_evidence.jsonl",
    },
    {
        "lang"       : "arabic",
        "train_path" : f"{BASE}/data/arabic/clef2026_gpt4_o_mini_train_arabic.json",
        "val_path"   : f"{BASE}/data/arabic/clef2026_gpt4_o_mini_val_arabic.json",
        "train_jsonl": f"{BASE}/output/training_data_for_RM/arabic_train_with_evidence.jsonl",
    },
]

# ── Evidence ablation ─────────────────────────────────────────────────────
# True  → Claim + Evidence + Verdict + Justification
# False → Claim + Verdict + Justification
EVIDENCE_CONDITIONS = [True, False]

# ── Aggregation strategies ────────────────────────────────────────────────
from collections import Counter

def agg_top1(verdict_list, score_list):
    return verdict_list[int(np.argmax(score_list))]

def agg_majority_top3(verdict_list, score_list):
    top_idx = np.argsort(score_list)[::-1][:min(3, len(verdict_list))]
    return Counter([verdict_list[i] for i in top_idx]).most_common(1)[0][0]

def agg_majority_top5(verdict_list, score_list):
    top_idx = np.argsort(score_list)[::-1][:min(5, len(verdict_list))]
    return Counter([verdict_list[i] for i in top_idx]).most_common(1)[0][0]

def agg_score_weighted(verdict_list, score_list):
    weights = torch.sigmoid(torch.tensor(score_list)).numpy()
    tally = {}
    for v, w in zip(verdict_list, weights):
        tally[v] = tally.get(v, 0.0) + float(w)
    return max(tally, key=tally.get)

AGGREGATIONS = [
    ("top1",           agg_top1),
    ("majority_top3",  agg_majority_top3),
    ("majority_top5",  agg_majority_top5),
    ("score_weighted", agg_score_weighted),
]

# ── Pre-create output dirs ────────────────────────────────────────────────
os.makedirs(f"{BASE}/output/training_data_for_RM", exist_ok=True)
os.makedirs(f"{BASE}/output/RM_prediction",        exist_ok=True)
os.makedirs("output/RM_prediction",                exist_ok=True)
for lc in LANGUAGES:
    for use_ev in EVIDENCE_CONDITIONS:
        ev_tag = "with_evidence" if use_ev else "no_evidence"
        os.makedirs(f"{BASE}/output/roberta_{ev_tag}_{lc['lang']}",         exist_ok=True)
        for agg_name, _ in AGGREGATIONS:
            os.makedirs(f"{BASE}/output/results_roberta_{ev_tag}_{lc['lang']}_{agg_name}", exist_ok=True)

print(f"\n{len(LANGUAGES)} language(s) × {len(EVIDENCE_CONDITIONS)} evidence condition(s) × {len(AGGREGATIONS)} aggregation(s)")
print(f"Total training runs : {len(LANGUAGES) * len(EVIDENCE_CONDITIONS)}")
print(f"Total scored runs   : {len(LANGUAGES) * len(EVIDENCE_CONDITIONS) * len(AGGREGATIONS)}")
for lc in LANGUAGES:
    print(f"  [{lc['lang']:8s}]  train={os.path.exists(lc['train_path'])}  val={os.path.exists(lc['val_path'])}")


Working dir: /content/drive/MyDrive/ColabNotebooks/AIR_Group_Task
Device: cuda

3 language(s) × 2 evidence condition(s) × 4 aggregation(s)
Total training runs : 6
Total scored runs   : 24
  [english ]  train=True  val=True
  [spanish ]  train=True  val=True
  [arabic  ]  train=True  val=True


In [6]:
# =========================
# 4. Helper functions
# =========================

def remove_label_pattern(text):
    text = re.sub(
        r"(\[?\s*Justification\s*\]?:?\s*)|(\[Label\]:\s*(True|False|Conflicting|Supports|Refutes))",
        "",
        text,
        flags=re.IGNORECASE,
    ).strip()
    return text.replace("\n", " ")


def get_evidence(sample):
    possible_keys = [
        "evidences",
        "evidence",
        "Evidence",
        "relevant_evidence",
        "Relevant_evidence",
        "context",
        "Context",
        "gold_evidence",
        "Gold_evidence",
    ]

    for key in possible_keys:
        if key in sample and sample[key]:
            value = sample[key]

            if isinstance(value, list):
                return " ".join(map(str, value))

            if isinstance(value, dict):
                return json.dumps(value, ensure_ascii=False)

            return str(value)

    return ""


def build_input(claim, evidence, verdict, justification):
    return (
        f"Claim: {claim}\n"
        f"Evidence: {evidence}\n"
        f"Verdict: {verdict}\n"
        f"Justification: {justification}"
    )


def build_input_no_evidence(claim, evidence, verdict, justification):
    return (
        f"Claim: {claim}\n"
        f"Verdict: {verdict}\n"
        f"Justification: {justification}"
    )


def print_trainable_parameters(model):
    trainable_params = 0
    all_params = 0

    for _, param in model.named_parameters():
        all_params += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()

    print(
        f"trainable params: {trainable_params} || "
        f"all params: {all_params} || "
        f"trainable%: {100 * trainable_params / all_params:.2f}"
    )


In [7]:
# =========================
# 5. Dataset
# =========================

class TextDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.texts = dataframe["model_input_text"].tolist()
        self.labels = dataframe["Class"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        item = {k: v.squeeze(0) for k, v in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)

        return item


In [8]:
# =========================
# 6. Roberta verifier model
# =========================

class CustomClassifier(torch.nn.Module):
    def __init__(
            self,
            model_name,
            num_labels=1,
            hidden_dim=None,
            dropout_value=0.1,
            freeze_base_layer=False,
    ):
        super().__init__()

        self.model = AutoModel.from_pretrained(model_name)

        if freeze_base_layer:
            for param in self.model.parameters():
                param.requires_grad = False

        hidden_size = self.model.config.hidden_size

        if hidden_dim:
            self.classifier = torch.nn.Sequential(
                torch.nn.Linear(hidden_size, hidden_dim),
                torch.nn.ReLU(),
                torch.nn.Dropout(dropout_value),
                torch.nn.Linear(hidden_dim, num_labels),
            )
        else:
            self.classifier = torch.nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        pooled_output = outputs.last_hidden_state[:, 0, :]
        pooled_output = pooled_output.float()

        logits = self.classifier(pooled_output)
        return logits

In [9]:
# =========================
# 7. Trainer
# =========================

class TrainerModule:
    def __init__(
            self,
            model,
            train_loader,
            val_loader,
            epochs,
            lr,
            output_dir,
    ):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = model.to(self.device)

        self.train_loader = train_loader
        self.val_loader = val_loader
        self.epochs = epochs

        self.optimizer = AdamW(self.model.parameters(), lr=lr, eps=1e-8)
        self.loss_fn = torch.nn.BCEWithLogitsLoss()

        total_steps = len(train_loader) * epochs
        warmup_steps = int(0.05 * total_steps)

        self.scheduler = get_cosine_schedule_with_warmup(
            self.optimizer,
            warmup_steps,
            total_steps,
        )

        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)

    def train(self):
        for epoch in range(self.epochs):
            print(f"\nEpoch {epoch + 1}/{self.epochs}")
            self.model.train()

            total_loss = 0
            total_acc = 0

            for batch in tqdm(self.train_loader):
                self.optimizer.zero_grad()

                input_ids = batch["input_ids"].to(self.device)
                attention_mask = batch["attention_mask"].to(self.device)
                labels = batch["labels"].to(self.device)

                logits = self.model(input_ids, attention_mask).squeeze(-1)
                logits = torch.clamp(logits, -50, 50)
                labels = labels.float()
                if torch.isnan(logits).any():
                  print("NaN detected — debugging batch")
                  print("input_ids max:", input_ids.max().item())
                  print("input_ids min:", input_ids.min().item())
                  print("attention_mask sum:", attention_mask.sum().item())
                  print("example text:", self.train_loader.dataset.texts[0][:500])
                  break
                assert not torch.isnan(logits).any(), "NaN in logits"
                loss = self.loss_fn(logits, labels)
                assert not torch.isnan(loss), "NaN loss"


                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                self.optimizer.step()
                self.scheduler.step()

                total_loss += loss.item()

                preds = (
                        torch.sigmoid(logits) >= 0.5
                ).detach().cpu().numpy()

                total_acc += accuracy_score(
                    labels.detach().cpu().numpy(),
                    preds,
                )

            print(f"Train Loss: {total_loss / len(self.train_loader):.4f}")
            print(f"Train Acc: {total_acc / len(self.train_loader):.4f}")

            self.evaluate(epoch)

    def evaluate(self, epoch):
        self.model.eval()

        total_loss = 0
        total_acc = 0

        with torch.no_grad():
            for batch in self.val_loader:
                input_ids = batch["input_ids"].to(self.device)
                attention_mask = batch["attention_mask"].to(self.device)
                labels = batch["labels"].to(self.device)

                logits = self.model(input_ids, attention_mask).squeeze(-1)
                logits = torch.clamp(logits, -50, 50)
                labels = labels.float()
                assert not torch.isnan(logits).any(), "NaN in logits"
                loss = self.loss_fn(logits, labels)
                assert not torch.isnan(loss), "NaN loss"

                total_loss += loss.item()

                preds = (
                        torch.sigmoid(logits) >= 0.5
                ).detach().cpu().numpy()

                total_acc += accuracy_score(
                    labels.detach().cpu().numpy(),
                    preds,
                )

        print(f"Val Loss: {total_loss / len(self.val_loader):.4f}")
        print(f"Val Acc: {total_acc / len(self.val_loader):.4f}")

        tokenizer.save_pretrained(self.output_dir)

        torch.save(
            self.model.state_dict(),
            os.path.join(self.output_dir, f"model_epoch_{epoch}.pt"),
        )


In [10]:
# =========================
# 9. Prediction evaluator
# =========================

class VerifierEvaluator:
    def __init__(
            self,
            model_path,
            tokenizer_path,
            base_model,
            device="cuda",
    ):
        self.device = torch.device(device if torch.cuda.is_available() else "cpu")

        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)

        self.model = CustomClassifier(
            model_name=base_model,
            freeze_base_layer=False,
        )

        self.model.load_state_dict(
            torch.load(model_path, map_location=self.device)
        )

        self.model.to(self.device)
        self.model.eval()

    def encode_input(
            self,
            claim,
            evidence,
            verdict,
            justification,
            max_length=MAX_LENGTH,
    ):
        text = build_input(
            claim=claim,
            evidence=evidence,
            verdict=verdict,
            justification=justification,
        )

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )

        return (
            encoding["input_ids"].to(self.device),
            encoding["attention_mask"].to(self.device),
        )

    def score(self, claim, evidence, verdict, justification):
        input_ids, attention_mask = self.encode_input(
            claim=claim,
            evidence=evidence,
            verdict=verdict,
            justification=justification,
        )

        with torch.no_grad():
            score = self.model(input_ids, attention_mask).item()

        return float(score)

In [12]:
# =========================
# 8. Main experiment loop
#    language × evidence_condition → train once, score once,
#    then apply all aggregations without re-running inference
# =========================

import re
all_results = []

for lang_cfg in LANGUAGES:
    lang = lang_cfg["lang"]

    print(f"\n{'='*60}")
    print(f"  LANGUAGE: {lang.upper()}")
    print(f"{'='*60}")

    # ── preprocessing (skip if JSONL already exists) ───────────────────────
    if os.path.exists(lang_cfg["train_jsonl"]):
        print(f"[{lang}] ✓ JSONL exists, skipping preprocessing")
    else:
        print(f"[{lang}] Running preprocessing ...")
        subprocess.run(
            [
                sys.executable,
                "experiments/teacher_llama_evidence/reasoning_trace_build_with_evidence.py",
                "--input",  lang_cfg["train_path"],
                "--output", lang_cfg["train_jsonl"],
            ],
            check=True,
        )

    base_train_df = pd.read_json(lang_cfg["train_jsonl"], lines=True)
    print(f"[{lang}] {len(base_train_df):,} training examples  |  class dist: {dict(base_train_df['Class'].value_counts())}")

    with open(lang_cfg["val_path"], "r", encoding="utf-8") as f:
        val_data = json.load(f)
    print(f"[{lang}] {len(val_data):,} val samples")

    for use_evidence in EVIDENCE_CONDITIONS:
        ev_tag   = "with_evidence" if use_evidence else "no_evidence"
        build_fn = build_input if use_evidence else build_input_no_evidence

        print(f"\n{'─'*60}")
        print(f"  [{lang}]  Evidence: {ev_tag.upper()}")
        print(f"{'─'*60}")

        MODEL_DIR = f"{BASE}/output/roberta_{ev_tag}_{lang}"
        os.makedirs(MODEL_DIR, exist_ok=True)

        # ── build model input text ─────────────────────────────────────────
        train_df = base_train_df.copy()
        train_df["model_input_text"] = [
            build_fn(
                claim=row["Claim"],
                evidence=row.get("Evidence", ""),
                verdict=row["Verdict"],
                justification=row["Justification"],
            )
            for _, row in train_df.iterrows()
        ]

        # ── train (skip if all checkpoints already exist) ──────────────────
        expected_ckpts = [
            os.path.join(MODEL_DIR, f"model_epoch_{e}.pt") for e in range(EPOCHS)
        ]
        if all(os.path.exists(p) for p in expected_ckpts):
            print(f"[{lang}|{ev_tag}] ✓ All {EPOCHS} checkpoints found, skipping training")
        else:
            train_split, dev_split = train_test_split(
                train_df, test_size=0.2,
                stratify=train_df["Class"], random_state=RANDOM_STATE,
            )
            tokenizer     = AutoTokenizer.from_pretrained(BASE_MODEL)
            train_dataset = TextDataset(train_split, tokenizer, MAX_LENGTH)
            dev_dataset   = TextDataset(dev_split,   tokenizer, MAX_LENGTH)
            train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
            dev_loader    = DataLoader(dev_dataset,   batch_size=BATCH_SIZE)

            model = CustomClassifier(model_name=BASE_MODEL, freeze_base_layer=False)
            print_trainable_parameters(model)

            trainer = TrainerModule(
                model=model, train_loader=train_loader, val_loader=dev_loader,
                epochs=EPOCHS, lr=LR, output_dir=MODEL_DIR,
            )
            trainer.train()

        # ── score all val traces once ──────────────────────────────────────
        print(f"[{lang}|{ev_tag}] Scoring validation traces ...")
        MODEL_PATH = os.path.join(MODEL_DIR, f"model_epoch_{EPOCHS - 1}.pt")
        evaluator  = VerifierEvaluator(
            model_path=MODEL_PATH,
            tokenizer_path=BASE_MODEL,
            base_model=BASE_MODEL,
        )

        scored_samples = []
        for idx, sample in enumerate(tqdm(val_data, desc=f"{lang}|{ev_tag}")):
            claim    = sample["claim"]
            evidence = get_evidence(sample)
            verdict_list, score_list, justification_list = [], [], []

            for trace_idx in range(len(sample["Reasoning_traces"])):
                justification = re.sub(
                    r"(\[?\s*Justification\s*\]?:?\s*)|(\[Label\]:\s*(True|False|Conflicting))",
                    "", sample["Reasoning_traces"][trace_idx], flags=re.IGNORECASE,
                ).strip().replace("\n", " ").split("Label:")[0]
                verdict = sample["Verdict_list"][trace_idx].lower()

                inp = build_fn(claim, evidence, verdict, justification)
                enc = evaluator.tokenizer(
                    inp, return_tensors="pt",
                    max_length=MAX_LENGTH, truncation=True, padding="max_length",
                )
                with torch.no_grad():
                    logits = evaluator.model(
                        enc["input_ids"].to(device),
                        enc["attention_mask"].to(device),
                    )
                score = logits[0][0].item()

                verdict_list.append(sample["Verdict_list"][trace_idx])
                justification_list.append(justification)
                score_list.append(score)

            scored_samples.append({
                "query_id"         : sample.get("query_id", idx),
                "Claim"            : claim,
                "Evidence"         : evidence,
                "Label"            : sample["label"],
                "verdict_list"     : verdict_list,
                "score_list"       : score_list,
                "justification_list": justification_list,
            })

        # ── apply each aggregation + run scorer ────────────────────────────
        for agg_name, agg_fn in AGGREGATIONS:
            print(f"  → [{lang}|{ev_tag}] aggregation: {agg_name}")

            RESULT_DIR = f"{BASE}/output/results_roberta_{ev_tag}_{lang}_{agg_name}"
            PRED_PATH  = f"{BASE}/output/RM_prediction/roberta_{ev_tag}_{lang}_{agg_name}_predictions.json"
            os.makedirs(RESULT_DIR, exist_ok=True)

            predictions = []
            for s in scored_samples:
                best_verdict = agg_fn(s["verdict_list"], s["score_list"])
                predictions.append({
                    "query_id"        : s["query_id"],
                    "Claim"           : s["Claim"],
                    "Evidence"        : s["Evidence"],
                    "Label"           : s["Label"],
                    "Verdict_BoN"     : best_verdict,
                    "BoN_Verdict_list": s["verdict_list"],
                    "Reasoning_traces": s["justification_list"],
                    "score_list"      : s["score_list"],
                })

            with open(PRED_PATH, "w", encoding="utf-8") as fp:
                json.dump(predictions, fp, indent=4, ensure_ascii=False)

            shutil.copy(PRED_PATH, "output/RM_prediction/clef_predictions.json")
            subprocess.run([sys.executable, "task2/scorer.py"], check=True)
            shutil.copy("output/RM_prediction/result.csv",        f"{RESULT_DIR}/result.csv")
            shutil.copy("output/RM_prediction/per_sample_ir.csv", f"{RESULT_DIR}/per_sample_ir.csv")

            # parse metrics from the non-standard CSV
            with open(f"{RESULT_DIR}/result.csv", "r") as f:
                content = f.read()
            import re as _re
            m_f1  = _re.search(r"^macro avg,[0-9.]+,[0-9.]+,([0-9.]+)", content, _re.MULTILINE)
            m_r5  = _re.search(r"^5,([0-9.]+)",                         content, _re.MULTILINE)
            macro_f1   = float(m_f1.group(1)) if m_f1 else float("nan")
            recall_at5 = float(m_r5.group(1)) if m_r5 else float("nan")

            all_results.append({
                "lang"       : lang,
                "evidence"   : ev_tag,
                "aggregation": agg_name,
                "macro_f1"   : macro_f1,
                "recall@5"   : recall_at5,
                "n_val"      : len(predictions),
            })
            print(f"     macro F1={macro_f1:.4f}  Recall@5={recall_at5:.4f}")



  LANGUAGE: ENGLISH
[english] ✓ JSONL exists, skipping preprocessing
[english] 31,433 training examples  |  class dist: {0: np.int64(22775), 1: np.int64(8658)}
[english] 1,600 val samples

────────────────────────────────────────────────────────────
  [english]  Evidence: WITH_EVIDENCE
────────────────────────────────────────────────────────────
[english|with_evidence] ✓ All 3 checkpoints found, skipping training
[english|with_evidence] Scoring validation traces ...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
english|with_evidence: 100%|██████████| 1600/1600 [04:46<00:00,  5.58it/s]


  → [english|with_evidence] aggregation: top1
     macro F1=0.4801  Recall@5=0.2548
  → [english|with_evidence] aggregation: majority_top3
     macro F1=0.4811  Recall@5=0.2548
  → [english|with_evidence] aggregation: majority_top5
     macro F1=0.4768  Recall@5=0.2548
  → [english|with_evidence] aggregation: score_weighted
     macro F1=0.4860  Recall@5=0.2548

────────────────────────────────────────────────────────────
  [english]  Evidence: NO_EVIDENCE
────────────────────────────────────────────────────────────


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

100%|██████████| 2021/2021 [04:36<00:00,  7.30it/s]


Train Loss: 0.5284
Train Acc: 0.7349
Val Loss: 0.5491
Val Acc: 0.7248
[arabic|no_evidence] Scoring validation traces ...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
arabic|no_evidence: 100%|██████████| 652/652 [02:31<00:00,  4.30it/s]


  → [arabic|no_evidence] aggregation: top1
     macro F1=0.5121  Recall@5=0.2424
  → [arabic|no_evidence] aggregation: majority_top3
     macro F1=0.5054  Recall@5=0.2424
  → [arabic|no_evidence] aggregation: majority_top5
     macro F1=0.4994  Recall@5=0.2424
  → [arabic|no_evidence] aggregation: score_weighted
     macro F1=0.5120  Recall@5=0.2424


In [13]:
# =========================
# 9. Results summary
# =========================

results_df = pd.DataFrame(all_results)

print("\n" + "=" * 70)
print("FULL RESULTS  (language x evidence x aggregation)")
print("=" * 70)
print(f"{'Language':<10} {'Evidence':<15} {'Aggregation':<16} {'Macro F1':>10} {'Recall@5':>10}")
print("-" * 70)
for _, row in results_df.sort_values(["lang", "evidence", "aggregation"]).iterrows():
    print(f"{row['lang']:<10} {row['evidence']:<15} {row['aggregation']:<16} "
          f"{row['macro_f1']:>10.4f} {row['recall@5']:>10.4f}")
print("=" * 70)

print("\nBEST CONFIG PER LANGUAGE (by macro F1):")
print("-" * 70)
for lang, grp in results_df.groupby("lang"):
    best = grp.loc[grp["macro_f1"].idxmax()]
    print(f"  {lang:<10}  evidence={best['evidence']:<15}  "
          f"agg={best['aggregation']:<16}  F1={best['macro_f1']:.4f}  R@5={best['recall@5']:.4f}")

print("\nEVIDENCE ABLATION (averaged across languages and aggregations):")
print("-" * 70)
for ev_tag, grp in results_df.groupby("evidence"):
    print(f"  {ev_tag:<16}  mean F1={grp['macro_f1'].mean():.4f}  mean R@5={grp['recall@5'].mean():.4f}")

print("\nAGGREGATION COMPARISON (averaged across languages and evidence):")
print("-" * 70)
for agg_name, grp in results_df.groupby("aggregation"):
    print(f"  {agg_name:<16}  mean F1={grp['macro_f1'].mean():.4f}  mean R@5={grp['recall@5'].mean():.4f}")

results_df.to_csv(f"{BASE}/output/roberta_all_results.csv", index=False)
print(f"\nSaved to: {BASE}/output/roberta_all_results.csv")



FULL RESULTS  (language x evidence x aggregation)
Language   Evidence        Aggregation        Macro F1   Recall@5
----------------------------------------------------------------------
arabic     no_evidence     majority_top3        0.5054     0.2424
arabic     no_evidence     majority_top5        0.4994     0.2424
arabic     no_evidence     score_weighted       0.5120     0.2424
arabic     no_evidence     top1                 0.5121     0.2424
arabic     with_evidence   majority_top3        0.4907     0.2102
arabic     with_evidence   majority_top5        0.4924     0.2102
arabic     with_evidence   score_weighted       0.4976     0.2102
arabic     with_evidence   top1                 0.4917     0.2102
english    no_evidence     majority_top3        0.5093     0.2977
english    no_evidence     majority_top5        0.4997     0.2977
english    no_evidence     score_weighted       0.5163     0.2977
english    no_evidence     top1                 0.5178     0.2977
english    with_evid

In [ ]:
# =========================
# 10. Prepare best runs for CLEF submission
# =========================

RUNS_DIR = f"{BASE}/runs"
os.makedirs(RUNS_DIR, exist_ok=True)

def convert_to_trec(pred_path, trec_path, run_tag):
    with open(pred_path, "r", encoding="utf-8") as f:
        predictions = json.load(f)
    with open(trec_path, "w", encoding="utf-8") as out:
        for sample in predictions:
            query_id = sample["query_id"]
            ranked   = sorted(enumerate(sample["score_list"]), key=lambda x: x[1], reverse=True)
            for rank, (trace_idx, score) in enumerate(ranked, start=1):
                out.write(f"{query_id}\tQ0\t{query_id}_{trace_idx}\t{rank}\t{score:.6f}\t{run_tag}\n")
    print(f"  Written {len(predictions)} queries → {trec_path}")

print("=" * 60)
print("BEST RUNS PER LANGUAGE")
print("=" * 60)

for lang, grp in results_df.groupby("lang"):
    best     = grp.loc[grp["macro_f1"].idxmax()]
    ev_tag   = best["evidence"]
    agg_name = best["aggregation"]
    src_pred = f"{BASE}/output/RM_prediction/roberta_{ev_tag}_{lang}_{agg_name}_predictions.json"
    run_tag  = f"roberta_{ev_tag}_{agg_name}"
    trec_path = os.path.join(RUNS_DIR, f"run_{lang}_{ev_tag}_{agg_name}.txt")

    print(f"\n  [{lang.upper()}]")
    print(f"    Config   : evidence={ev_tag}  aggregation={agg_name}")
    print(f"    Macro F1 : {best['macro_f1']:.4f}  Recall@5: {best['recall@5']:.4f}")

    if not os.path.exists(src_pred):
        print(f"    ✗ Prediction file not found!")
        continue

    shutil.copy(src_pred, os.path.join(RUNS_DIR, f"run_{lang}_{ev_tag}_{agg_name}.json"))
    convert_to_trec(src_pred, trec_path, run_tag)
    print(f"    TREC file: {trec_path}")

BEST RUNS PER LANGUAGE

  [ARABIC]
    Config   : evidence=no_evidence  aggregation=top1
    Macro F1 : 0.5121  Recall@5: 0.2424
  Written 652 queries → /content/drive/MyDrive/ColabNotebooks/AIR_Group_Task/runs/run_arabic_no_evidence_top1.txt
    TREC file: /content/drive/MyDrive/ColabNotebooks/AIR_Group_Task/runs/run_arabic_no_evidence_top1.txt

  [ENGLISH]
    Config   : evidence=no_evidence  aggregation=top1
    Macro F1 : 0.5178  Recall@5: 0.2977
  Written 1600 queries → /content/drive/MyDrive/ColabNotebooks/AIR_Group_Task/runs/run_english_no_evidence_top1.txt
    TREC file: /content/drive/MyDrive/ColabNotebooks/AIR_Group_Task/runs/run_english_no_evidence_top1.txt

  [SPANISH]
    Config   : evidence=no_evidence  aggregation=top1
    Macro F1 : 0.4354  Recall@5: 0.2754
  Written 562 queries → /content/drive/MyDrive/ColabNotebooks/AIR_Group_Task/runs/run_spanish_no_evidence_top1.txt
    TREC file: /content/drive/MyDrive/ColabNotebooks/AIR_Group_Task/runs/run_spanish_no_evidence_top1